In [1]:
import os
import pickle

import numpy as np
from tqdm import tqdm

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model

In [2]:
BASE_DIR = "../artifacts/Flickr8k"
IMAGES_DIR = os.path.join(BASE_DIR, "Images")
ARTIFACTS_DIR = "../artifacts" 

In [3]:
vgg = VGG16()

In [ ]:
#keep it up to the fc2 layer

feature_model = Model(inputs=vgg.inputs, outputs=vgg.layers[-2].output)

In [6]:
#output layer
feature_model.layers[-1].name

'fc2'

In [8]:
#shape of output layer
feature_model.output_shape

(None, 4096)

In [9]:
def extract_features(img_path, model):
    image = load_img(img_path, target_size=(224, 224))  # 
    image = img_to_array(image)                         
    image = image.reshape(1, 224, 224, 3)               
    image = preprocess_input(image)                    
    feature = model.predict(image, verbose=0)          
    return feature

In [10]:
sample_path = os.path.join(IMAGES_DIR, "1000268201_693b08cb0e.jpg")

In [11]:
sample_feature = extract_features(sample_path, feature_model)

/opt/anaconda3/envs/imagecaption/lib/python3.10/site-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


In [14]:
sample_feature

array([[2.507474, 0.      , 0.      , ..., 0.      , 0.      , 0.      ]],
      dtype=float32)

In [13]:
sample_feature.shape

(1, 4096)

## Extract features for every image

This loops over all 8,091 images. On CPU it takes a few minutes. We sort the
filenames so the extraction order is the same on every run.

In [15]:
features = {}

for image_name in tqdm(sorted(os.listdir(IMAGES_DIR))):
    if not image_name.lower().endswith(".jpg"):
        continue
    img_path = os.path.join(IMAGES_DIR, image_name)
    image_id = image_name.split(".")[0]
    features[image_id] = extract_features(img_path, feature_model)

100%|██████████| 8091/8091 [20:45<00:00,  6.49it/s]


In [19]:
features

{'1000268201_693b08cb0e': array([[2.507474, 0.      , 0.      , ..., 0.      , 0.      , 0.      ]],
       dtype=float32),
 '1001773457_577c3a7d70': array([[0.        , 0.        , 0.49414793, ..., 0.        , 0.        ,
         0.        ]], dtype=float32),
 '1002674143_1b742ab4b8': array([[1.4937913, 0.       , 0.5356748, ..., 2.3152368, 3.7418268,
         0.       ]], dtype=float32),
 '1003163366_44323f5815': array([[0., 0., 0., ..., 0., 0., 0.]], dtype=float32),
 '1007129816_e794419615': array([[0.        , 0.09227663, 0.        , ..., 0.        , 0.        ,
         0.06529236]], dtype=float32),
 '1007320043_627395c3d8': array([[0.      , 0.      , 0.      , ..., 0.      , 3.339018, 0.      ]],
       dtype=float32),
 '1009434119_febe49276a': array([[2.096517  , 2.1192532 , 3.5619893 , ..., 0.64242965, 2.7142005 ,
         0.        ]], dtype=float32),
 '1012212859_01547e3f17': array([[0.        , 0.        , 0.9873728 , ..., 0.        , 1.4932494 ,
         0.86128426]], dty

In [16]:
len(features)

8091

In [17]:
sample = next(iter(features.values()))
print("shape   :", sample.shape)
print("min/max :", float(sample.min()), "/", float(sample.max()))

assert sample.shape == (1, 4096), "unexpected feature shape"
assert sample.min() >= 0, "fc2 features should never be negative"
print("Sanity check passed.")

shape   : (1, 4096)
min/max : 0.0 / 8.961539268493652
Sanity check passed.


## Save the features

In [18]:
with open(os.path.join(ARTIFACTS_DIR, "features.pkl"), "wb") as f:
    pickle.dump(features, f)

print("Saved ../artifacts/features.pkl")

Saved ../artifacts/features.pkl


Next: **05_data_generator** - feed image features + caption sequences to the
model in batches.